# Exploratory Data Analysis

Inspect the full `merged_order_items` table from `data/interim`. One row represents an order item, identified by `(order_id, order_item_id)`.

The workflow covers structure, distributions, missingness, domain checks and detailed inspection of unusual records. All observations retain their original values and statuses. Diagnostic tables and figures remain in memory; this notebook writes no datasets or documentation.

Run all cells in order. Section functions in `src.eda.workflow` display their evidence and return complete result objects for further inspection. Cleaning, feature engineering and delivery-performance analysis are separate workflows.


## 1. Input and execution context

### Repository location

Locate the repository from the current working directory before importing project modules. Execution is supported from the repository root or its `notebooks` directory.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


# Locate the repository before importing project modules.
def find_project_root(start_path: Path) -> Path:
    """Find the project root containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.data_loader import load_data
from src.eda.checks import build_entity_views, profile_entity_consistency, profile_keys
from src.eda.profiling import get_column_roles, profile_columns, profile_dataset
from src.eda.workflow import (
    inspect_numeric_distributions,
    inspect_category_distributions,
    inspect_temporal_distributions,
    inspect_missing_patterns,
    inspect_missing_pairs,
    inspect_conditional_missingness,
    inspect_status_missingness,
    inspect_text_formats,
    inspect_order_timeline,
    inspect_measurement_plausibility,
    inspect_order_structure,
    inspect_timeline_details,
    inspect_delivered_completeness,
    inspect_product_completeness,
    inspect_numeric_tails,
    inspect_deadline_calendar,
    inspect_seller_city_spacing,
    inspect_structure_overlap,
)
from src.paths import INTERIM_DIR
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")


### Load the full merged table

The schema-aware reader tries Parquet first and falls back to CSV. Set `file_format="csv"` to read CSV directly.

The in-memory baseline records shape, column order, dtypes and indexed row hashes for the final input-preservation assertion.


In [ ]:
df = load_data("merged_order_items", stage="interim")

# Record the loaded input without making a second copy of the full dataset.
input_baseline = {
    "shape": df.shape,
    "columns": df.columns.copy(),
    "dtypes": df.dtypes.copy(),
    "row_hashes": pd.util.hash_pandas_object(df, index=True).copy(),
}


### Reference dimensions

Compare observed dimensions with the reference full merge. Reference counts identify the input version; they are not eligibility conditions or cleaning targets.


In [ ]:
baseline_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
        ["Unique orders", 98_666, df["order_id"].nunique()],
    ],
    columns=["Metric", "Reference", "Observed"],
)
baseline_summary["Matches reference"] = baseline_summary["Reference"].eq(
    baseline_summary["Observed"]
)
display(baseline_summary)

if not baseline_summary["Matches reference"].all():
    print("Reference counts differ. Check the input version before interpreting later profiles.")

print("Input baseline retained in memory for the final non-mutation check.")


### Data preview

Display the first item rows with their original column values.


In [ ]:
df.head()

## 2. Structure and completeness

### Dataset dimensions

Full-table statistics use item rows. Exact duplicate participants count every row in a repeated-row group; excess copies exclude the first occurrence.

The merged table contains `customer_id`, which identifies a customer record attached to a purchase. It does not contain `customer_unique_id`; customer-record counts therefore do not represent unique people.


In [ ]:
dataset_profile = profile_dataset(df)
display(dataset_profile)


### Column roles and completeness

Column roles come from the merge schema. Source ownership describes the contributing table and its key. Identifiers and ZIP prefixes are excluded from numeric measurements.

Missing values use `isna()`. Empty or whitespace-only strings are counted separately as blanks. Percentages use all item rows; distinct non-null counts preserve original case, whitespace and blank strings.


In [ ]:
column_roles = get_column_roles()
column_profile = profile_columns(df)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Registered column roles")
    display(column_roles)
    print("Completeness at item grain")
    display(column_profile)


### Keys and repeated identifiers

The full-table key is `(order_id, order_item_id)`. Repeated order, customer, product and seller IDs are expected at item grain.

`invalid_rows` counts the union of missing and blank key rows. Distinct-key and repeated-key statistics use valid keys. `excess_key_rows` counts valid rows beyond the first occurrence of each key.


In [ ]:
key_profile = profile_keys(df)
with pd.option_context("display.max_columns", None):
    display(key_profile)


### Entity attribute consistency

Check each source attribute within its order, customer, product or seller ID. A null paired with a non-null value is a conflict. An attribute that is null throughout an entity remains consistent but incomplete.

Invalid IDs are counted separately. Conflict percentages use valid distinct entities.


In [ ]:
entity_consistency = profile_entity_consistency(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Consistency by entity")
    display(entity_consistency["summary"])
    print("Consistency by attribute")
    display(entity_consistency["attributes"])

if entity_consistency["conflicts"].empty:
    print("No conflicting entity attributes found.")
else:
    print("Conflicting entity attributes (all detected entity/attribute pairs)")
    display(entity_consistency["conflicts"])


### Source-grain views

Construct one diagnostic row per observed source key after validating key completeness and attribute consistency. Invalid keys or conflicts raise an error.

These views cover entities present in the full merge. They preserve source dtypes and leave the full item table unchanged.


In [ ]:
entity_views = build_entity_views(df)
entity_view_summary = pd.DataFrame(
    [
        [name, len(view), len(view.columns)]
        for name, view in entity_views.items()
    ],
    columns=["View", "Observed rows", "Columns"],
)
display(entity_view_summary)


## 3. Distributions

### Numeric measurements

Price and freight use item grain (BRL). Weight and dimensions use product grain (grams and centimetres).

Statistics use finite non-missing values; missing and non-finite counts are reported separately. Standard deviation uses `ddof=1`. Each measurement has a full-range histogram, a 1st–99th percentile display window and a full-range boxplot. The window changes only the plotted range. IQR limits describe the observed distribution.


In [ ]:
numeric_profiles = inspect_numeric_distributions(
    entity_views=entity_views,
    column_roles=column_roles,
)


### Categories and ZIP prefixes

Status uses order grain; customer and seller locations use their respective entity grain; product category uses product grain.

Complete frequency tables remain available in `category_distributions["frequency_tables"]`. Displays show the top 15 values plus Other, Blank and Missing. Counts retain the full population and original text forms.


In [ ]:
category_distributions = inspect_category_distributions(
    entity_views=entity_views,
    column_roles=column_roles,
)


### Timestamp coverage and calendar distributions

Order timestamps use one row per order. Shipping deadlines use item grain. Coverage includes missingness, range and recorded clock precision; calendar profiles cover month, date, weekday and hour.

Calendar components use the stored clock. Month and date tables contain observed buckets; weekday and hour tables include all seven and 24 buckets. Missing timestamps remain in the all-row denominator. Only date displays are limited to the top 15 dates plus Other and Missing; complete frequencies remain in `temporal_distributions`.


In [ ]:
temporal_distributions = inspect_temporal_distributions(
    entity_views=entity_views,
    column_roles=column_roles,
)


## 4. Missingness in context

### Exact patterns and source-grain completeness

Inspect missingness in the 25-column item table and in order, product, customer and seller views. Each row contributes once to a missing-count bin and once to an exact pattern; `()` denotes a complete row within the selected columns.

Percentages use all rows of the named view. Complete result tables, including zero-count bins, remain in `missing_patterns`. Blanks remain separate from nulls.


In [ ]:
missing_patterns = inspect_missing_patterns(
    df=df,
    entity_views=entity_views,
    column_profile=column_profile,
)


### Paired and directional missingness

Retain every column pair for full items, orders and products. Display pairs involving columns with observed missingness, including pairs with zero jointly missing rows.

Pair counts partition the population into four states. Heatmap rows condition on a column being missing; columns show the percentage also missing another column. Undefined denominators remain NA. Directional percentages may differ.


In [ ]:
missing_pair_profiles = inspect_missing_pairs(
    missingness_frames=missing_patterns["missingness_frames"],
    native_column_profiles=missing_patterns["native_column_profiles"],
)


### Missingness within observed groups

Group order-timestamp missingness by status, purchase month and customer state. Group product completeness by category and item-level exposure by seller and geography. Purchase month is a temporary calendar grouping.

Missing grouping keys remain included. Each table retains group size, evaluated, unknown and affected counts with explicit denominators. Full groups remain in `missing_context["condition_profiles"]`; limited displays state how many groups are shown.


In [ ]:
missing_context = inspect_conditional_missingness(
    column_roles=column_roles,
    df=df,
    entity_views=entity_views,
)


### Timestamp missingness by status

Each heatmap cell uses all orders in its status as the denominator. Columns cover individual timestamps and the union of the four core timestamps. This differs from the missing-column denominator in the directional heatmaps.


In [ ]:
status_missing_rates = inspect_status_missingness(
    condition_profiles=missing_context["condition_profiles"],
    order_timestamp_columns=missing_context["order_timestamp_columns"],
)


## 5. Domain consistency

### Text, identifiers and location codes

Inspect original strings for whitespace, control/replacement characters, type and structural patterns. String IDs are checked against 32 lowercase hexadecimal characters, ZIP prefixes against five ASCII digits and states against two uppercase letters. These are format checks, not address validation.

Regex percentages use evaluated string rows. Temporary stripped/casefolded keys identify candidate spelling variants without modifying or consolidating values. Examples retain original row positions; complete tables remain in `text_format_profiles`.


In [ ]:
text_format_profiles = inspect_text_formats(
    column_roles=column_roles,
    missingness_frames=missing_patterns["missingness_frames"],
)


### Order timestamp sequence

Evaluate six physical timestamp comparisons and one estimated-date comparison. Comparisons require both timestamps; unavailable comparisons remain NA. Percentages use comparable orders, and flags can overlap.

Physical comparisons retain clock precision. Estimated delivery before purchase uses calendar dates. Equal values are not flagged. Results include original examples and status-specific counts.


In [ ]:
order_timeline = inspect_order_timeline(
    entity_views=entity_views,
)


### Measurements and item deadlines

Inspect zero, negative, missing and non-finite measurements using the numeric profiles. Zero freight is counted separately.

Compare item deadlines with purchase timestamps where both exist. The nullable condition remains at item grain and does not select an order-level deadline or assign seller delay.


In [ ]:
measurement_checks = inspect_measurement_plausibility(
    numeric_profiles=numeric_profiles,
    df=df,
)


### Order structure

Count items, products, sellers, distinct deadlines and observed state pairs within each order. Consecutive item numbering requires unique positive integer IDs covering `1..N`.

State equality uses present, non-blank original strings. Missing deadlines and incomparable states are counted separately. Order-seller deadline ranges retain the observed values without selecting a representative deadline.


In [ ]:
order_structure = inspect_order_structure(
    df=df,
)


## 6. Detailed record investigations

### Timestamp reversal precision and overlap

Separate observed ordering flags into same-day and earlier-calendar-date reversals. Retain overlapping flag combinations and bounded original examples.

Monthly tables use comparable orders within each purchase month. The two monthly plots show counts among flagged orders; their percentages describe the flagged cohort.


In [ ]:
timeline_details = inspect_timeline_details(
    entity_views=entity_views,
    timeline_profile=order_timeline["timeline_profile"],
    order_context=missing_context["order_context"],
)


### Core timestamps in delivered orders

Inspect a temporary delivered-order cohort. Retain exact missing-core patterns, original records, purchase-month groups and the overlap with observed ordering flags.

The core set contains purchase, approval, carrier handoff and customer delivery. All statuses remain in the full input.


In [ ]:
delivered_completeness = inspect_delivered_completeness(
    orders=entity_views["orders"],
    core_timestamp_columns=missing_context["core_timestamp_columns"],
    any_ordering_flag=timeline_details["any_ordering_flag"],
    order_context=missing_context["order_context"],
)


### Physical measurements and repeated product exposure

Separate missing physical measurements from nonpositive measurements. Report affected products, their item occurrences and distinct orders with separate denominators.

Compare category/physical co-missingness at product and item grain. Inspect zero freight by status and category without changing values.


In [ ]:
product_completeness = inspect_product_completeness(
    entity_views=entity_views,
    product_measurement_columns=missing_context["product_measurement_columns"],
    df=df,
    orders=entity_views["orders"],
)


### Numeric tails and category composition

Inspect the existing 1.5-IQR tails by product category. Price and freight use item grain; physical measurements use product grain. Retain finite and unassessable denominators.

Complete category tables remain in `numeric_tails`. Displays show up to 15 groups and up to three original records from each numeric extreme.


In [ ]:
numeric_tails = inspect_numeric_tails(
    numeric_profiles=numeric_profiles,
    df=df,
    entity_views=entity_views,
)


### Deadlines beyond the purchase observation years

Compare shipping-deadline years with the latest observed purchase year. Display original item deadlines, lifecycle timestamps and statuses, with separate item, order and order-seller counts.

The observation-year boundary is a descriptive reference; it does not establish an SLA or an invalid-date rule.


In [ ]:
deadline_calendar = inspect_deadline_calendar(
    orders=entity_views["orders"],
    df=df,
    order_timestamp_columns=missing_context["order_timestamp_columns"],
)


### Repeated spaces in seller cities

Inspect affected seller records and their item/order exposure. Compare original city spellings within the same state using temporary collapsed-whitespace/casefolded keys.

`[space]` makes repeated spaces visible in display columns. Original strings remain unchanged; comparison keys do not establish geographic identity.


In [ ]:
seller_city_spacing = inspect_seller_city_spacing(
    entity_views=entity_views,
    df=df,
    orders=entity_views["orders"],
)


### Overlap of seller, deadline and state-pair multiplicity

Retain every observed combination of multiple sellers, multiple deadlines and mixed equal/different state pairs at order grain. Inspect status groups and original item rows for bounded example orders.

These tables describe order composition without assigning a route or choosing a representative seller or deadline.


In [ ]:
structure_overlap = inspect_structure_overlap(
    order_composition=order_structure["order_composition"],
    orders=entity_views["orders"],
    df=df,
)


## 7. Input preservation

Assert that the full input still matches its initial shape, column order, dtypes, indexed values and row order. The assertion writes no files.


In [ ]:
assert df.shape == input_baseline["shape"], "Input shape changed during profiling."
pd.testing.assert_index_equal(df.columns, input_baseline["columns"])
pd.testing.assert_series_equal(df.dtypes, input_baseline["dtypes"])
pd.testing.assert_series_equal(
    pd.util.hash_pandas_object(df, index=True), input_baseline["row_hashes"],
)
print("Input preserved after complete pre-cleaning EDA.")